In [15]:
import kagglehub
import pandas as pd
import os

## Load Dataset

[Dataset](https://www.kaggle.com/datasets/vbookshelf/respiratory-sound-database)


In [2]:
path = kagglehub.dataset_download("vbookshelf/respiratory-sound-database")
print("Path to dataset files:", path)

100%|██████████| 3.69G/3.69G [03:16<00:00, 20.1MB/s]

Extracting files...


Path to dataset files: C:\Users\crick\.cache\kagglehub\datasets\vbookshelf\respiratory-sound-database\versions\2


In [ ]:
os.listdir(path)


['demographic_info.txt', 'Respiratory_Sound_Database']

In [10]:
os.listdir(path+"\Respiratory_Sound_Database\Respiratory_Sound_Database")

['audio_and_txt_files',
 'filename_differences.txt',
 'filename_format.txt',
 'patient_diagnosis.csv']

In [9]:
path =  "C:\\Users\\crick\\.cache\\kagglehub\\datasets\\vbookshelf\\respiratory-sound-database\\versions\\2"
audio_files_path = path+"\\Respiratory_Sound_Database\\Respiratory_Sound_Database\\audio_and_txt_files"
diagnosis_csv_path = path + "\\Respiratory_Sound_Database\\Respiratory_Sound_Database"

## EDA

In [16]:
column_names = ['ID', 'Age', 'Sex', 'BMI', 'Weight_kg', 'Height_cm']
df_demographics = pd.read_csv(f'{path}/demographic_info.txt', sep='\s+', header=None, names=column_names)
display(df_demographics.head())

,ID,Age,Sex,BMI,Weight_kg,Height_cm
0,101,3.00,F,NaN,19.0,99.0
1,102,0.75,F,NaN,9.8,73.0
2,103,70.00,F,33.00,NaN,NaN
3,104,70.00,F,28.47,NaN,NaN
4,105,7.00,F,NaN,32.0,135.0


In [17]:
df_demographics.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 6 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   ID         126 non-null    int64  
 1   Age        125 non-null    float64
 2   Sex        125 non-null    str    
 3   BMI        75 non-null     float64
 4   Weight_kg  44 non-null     float64
 5   Height_cm  42 non-null     float64
dtypes: float64(4), int64(1), str(1)
memory usage: 6.0 KB


In [18]:
df_diag = pd.read_csv(f'{diagnosis_csv_path}\patient_diagnosis.csv',names=['ID','Diagnosis'])
df_diag.head()

,ID,Diagnosis
0,101,URTI
1,102,Healthy
2,103,Asthma
3,104,COPD
4,105,URTI


In [19]:
df_diag.info()

<class 'pandas.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 2 columns):
 #   Column     Non-Null Count  Dtype
---  ------     --------------  -----
 0   ID         126 non-null    int64
 1   Diagnosis  126 non-null    str  
dtypes: int64(1), str(1)
memory usage: 2.1 KB


In [20]:
df_diag.Diagnosis.value_counts()

Diagnosis
COPD              64
Healthy           26
URTI              14
Bronchiectasis     7
Pneumonia          6
Bronchiolitis      6
LRTI               2
Asthma             1
Name: count, dtype: int64

## Visualize the sounds

In [22]:
import random
import os

# Load diagnosis lookup
df_diag = pd.read_csv(f'{diagnosis_csv_path}\\patient_diagnosis.csv', names=['ID', 'Diagnosis'])
diag_map = dict(zip(df_diag['ID'], df_diag['Diagnosis']))

def parse_annotation(txt_path):
    """Parse .txt annotation file → DataFrame with columns [start, end, crackles, wheezes]."""
    return pd.read_csv(txt_path, sep='\t', header=None,
                       names=['start', 'end', 'crackles', 'wheezes'])

def decode_filename(fname):
    """Extract metadata from recording filename."""
    parts = os.path.splitext(fname)[0].split('_')
    pid = int(parts[0])
    location_map = {
        'Tc': 'Trachea', 'Al': 'Anterior Left', 'Ar': 'Anterior Right',
        'Pl': 'Posterior Left', 'Pr': 'Posterior Right',
        'Ll': 'Lateral Left', 'Lr': 'Lateral Right'
    }
    equipment_map = {
        'Meditron': 'WelchAllyn Meditron', 'LittC2SE': '3M Littmann Classic II SE',
        'Litt3200': '3M Littmann 3200',    'AKGC417L': 'AKG C417L Mic'
    }
    return pid, location_map.get(parts[2], parts[2]), \
           ('Single-channel' if parts[3]=='sc' else 'Multi-channel'), \
           equipment_map.get(parts[4], parts[4])


In [23]:
from IPython.display import display

all_wav = [f for f in os.listdir(audio_files_path) if f.endswith('.wav')]

random.seed(42)
sampled = random.sample(all_wav, 4)
print(f"Total WAV files: {len(all_wav)}\nSampled: {sampled}\n")

for i, wav_file in enumerate(sampled, 1):
    pid, location, mode, equipment = decode_filename(wav_file)
    diagnosis = diag_map.get(pid, 'Unknown')

    # Load annotation
    df_ann = parse_annotation(os.path.join(audio_files_path, wav_file.replace('.wav', '.txt')))
    n_crackles = int(df_ann['crackles'].sum())
    n_wheezes  = int(df_ann['wheezes'].sum())
    n_cycles   = len(df_ann)

    # Load audio
    audio, sr = librosa.load(os.path.join(audio_files_path, wav_file), sr=22050)

    print(f"{'='*60}")
    print(f"Sample {i}: {wav_file}")
    print(f"  Patient: {pid}  |  Diagnosis : {diagnosis}")
    print(f"  Location: {location}  |  Mode: {mode}  |  Equipment: {equipment}")
    print(f"  Duration: {df_ann['end'].iloc[-1]:.2f}s  |  Breath cycles: {n_cycles}")
    print(f"  Crackles: {n_crackles}/{n_cycles} cycles  ({'⚠ PRESENT' if n_crackles else '✓ Absent'})")
    print(f"  Wheezes : {n_wheezes}/{n_cycles} cycles  ({'⚠ PRESENT' if n_wheezes  else '✓ Absent'})")
    print(f"\n  Cycle annotations:\n{df_ann.to_string(index=False)}\n")
    print("  ▶ Audio:")
    display(ipd.Audio(audio, rate=sr))
    print()


Total WAV files: 920
Sampled: ['186_2b2_Al_mc_AKGC417L.wav', '130_1p2_Lr_mc_AKGC417L.wav', '107_2b4_Pr_mc_AKGC417L.wav', '201_1b1_Ar_sc_Meditron.wav']

Sample 1: 186_2b2_Al_mc_AKGC417L.wav
  Patient: 186  |  Diagnosis : COPD
  Location: Anterior Left  |  Mode: Multi-channel  |  Equipment: AKG C417L Mic
  Duration: 18.46s  |  Breath cycles: 5
  Crackles: 0/5 cycles  (✓ Absent)
  Wheezes : 1/5 cycles  (⚠ PRESENT)

  Cycle annotations:
 start    end  crackles  wheezes
 1.351  5.292         0        1
 5.292  9.423         0        0
 9.423 13.006         0        0
13.006 17.125         0        0
17.125 18.458         0        0

  ▶ Audio:



Sample 2: 130_1p2_Lr_mc_AKGC417L.wav
  Patient: 130  |  Diagnosis : COPD
  Location: Lateral Right  |  Mode: Multi-channel  |  Equipment: AKG C417L Mic
  Duration: 19.55s  |  Breath cycles: 7
  Crackles: 7/7 cycles  (⚠ PRESENT)
  Wheezes : 3/7 cycles  (⚠ PRESENT)

  Cycle annotations:
 start    end  crackles  wheezes
 0.697  2.423         1        0
 2.423  5.101         1        0
 5.101  8.339         1        0
 8.339 11.280         1        0
11.280 14.316         1        1
14.316 17.030         1        1
17.030 19.554         1        1

  ▶ Audio:



Sample 3: 107_2b4_Pr_mc_AKGC417L.wav
  Patient: 107  |  Diagnosis : COPD
  Location: Posterior Right  |  Mode: Multi-channel  |  Equipment: AKG C417L Mic
  Duration: 19.54s  |  Breath cycles: 8
  Crackles: 8/8 cycles  (⚠ PRESENT)
  Wheezes : 0/8 cycles  (✓ Absent)

  Cycle annotations:
 start    end  crackles  wheezes
 1.018  3.411         1        0
 3.411  5.827         1        0
 5.827  8.339         1        0
 8.339 10.923         1        0
10.923 13.292         1        0
13.292 16.018         1        0
16.018 18.482         1        0
18.482 19.542         1        0

  ▶ Audio:



Sample 4: 201_1b1_Ar_sc_Meditron.wav
  Patient: 201  |  Diagnosis : Bronchiectasis
  Location: Anterior Right  |  Mode: Single-channel  |  Equipment: WelchAllyn Meditron
  Duration: 19.95s  |  Breath cycles: 7
  Crackles: 0/7 cycles  (✓ Absent)
  Wheezes : 6/7 cycles  (⚠ PRESENT)

  Cycle annotations:
 start    end  crackles  wheezes
 0.022  0.893         0        0
 0.893  4.093         0        1
 4.093  7.250         0        1
 7.250 10.364         0        1
10.364 13.707         0        1
13.707 17.107         0        1
17.107 19.950         0        1

  ▶ Audio:
